## ML_1048: GRPO (Group Relative Policy Optimization) Loss

**Difficulty**: Hard | **TorchCode**: #38

### Problem
Implement the GRPO loss used in DeepSeek-R1. For each group of responses to the same prompt, normalize rewards into advantages via z-score within the group, then compute the policy gradient loss.

### Formula
$$A_i = \frac{r_i - \mu_g}{\sigma_g + \varepsilon}, \quad \mathcal{L}_{\text{GRPO}} = -\mathbb{E}\!\left[\text{sg}(A_i) \cdot \log\pi_\theta(y_i)\right]$$

In [ ]:
import torch
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)
    for gid in unique_ids:
        mask = group_ids == gid
        r_g = rewards[mask]
        advantages[mask] = (r_g - r_g.mean()) / (r_g.std(unbiased=False) + eps)
    return -(advantages.detach() * logps).mean()

In [ ]:
import torch
from torch import Tensor

# --- Test 1: Basic shape & type ---
logps = torch.randn(6, requires_grad=True)
rewards = torch.randn(6)
group_ids = torch.tensor([0, 0, 0, 1, 1, 1])
loss = grpo_loss(logps, rewards, group_ids)
assert isinstance(loss, Tensor) and loss.dim() == 0

# --- Test 2: Gradient flows to logps only ---
logps2 = torch.randn(4, requires_grad=True)
rewards2 = torch.randn(4, requires_grad=True)
group_ids2 = torch.tensor([0, 0, 1, 1])
grpo_loss(logps2, rewards2, group_ids2).backward()
assert logps2.grad is not None and rewards2.grad is None

# --- Test 3: Group-wise normalization is independent ---
logps3 = torch.zeros(4, requires_grad=True)
rewards3 = torch.tensor([0.0, 1.0, 10.0, 11.0])
group_ids3 = torch.tensor([0, 0, 1, 1])
grpo_loss(logps3, rewards3, group_ids3).backward()
assert torch.allclose(logps3.grad[:2], logps3.grad[2:])

# --- Test 4: Numerical correctness ---
logps4 = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards4 = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids4 = torch.tensor([0, 0, 1, 1])
loss = grpo_loss(logps4, rewards4, group_ids4)
# Manual reference
A0 = (rewards4[:2] - rewards4[:2].mean()) / (rewards4[:2].std(unbiased=False) + 1e-5)
A1 = (rewards4[2:] - rewards4[2:].mean()) / (rewards4[2:].std(unbiased=False) + 1e-5)
A = torch.cat([A0, A1])
ref = -(A * logps4).mean()
assert torch.allclose(loss, ref, atol=1e-4)

print('All tests passed!')